In [24]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [25]:
load_dotenv()

model = ChatOpenAI()

In [26]:
class BlogState(TypedDict):

    title:str
    outline: str
    content: str
    evaluation: str
    is_approved: bool     # pass/fail
    iteration: int        # retry counter

In [27]:
def create_outline(state:BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [29]:
def evaluate_blog(state: BlogState) -> BlogState:
    
    content = state['content']

    prompt = f'''Evaluate the following blog for clarity, structure, and quality.
    Respond with "APPROVED" on the first line if it's good.
    Otherwise respond with "REJECTED" on the first line followed by specific feedback on what to fix.

    Blog:
    {content}'''

    result = model.invoke(prompt).content

    state['is_approved'] = result.strip().startswith('APPROVED')
    state['evaluation'] = result
    state['iteration'] = state.get('iteration', 0) + 1

    return state

In [30]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']
    feedback = state.get('evaluation', '')

    prompt = f'Write a detailed blog on the title - {title} using the following outline \n {outline}'
    if feedback:
        prompt += f'\n\nPrevious attempt was rejected with this feedback, address it:\n{feedback}'

    content = model.invoke(prompt).content   # note: .content, not the message object

    state['content'] = content

    return state

In [31]:
MAX_ITER = 3

def route_evaluation(state: BlogState) -> str:
    if state['is_approved'] or state['iteration'] >= MAX_ITER:
        return 'end'
    return 'retry'

In [ ]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)


# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

In [15]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

print(final_state['outline'])

{'title': 'Rise of AI in India', 'outline': "I. Introduction\n    A. Definition of AI\n    B. Overview of the rise of AI in India\n    C. Importance of AI in today's world\n   \nII. Historical Background of AI in India\n    A. Introduction of AI in India\n    B. Evolution of AI technology in India\n    C. Key milestones in the development of AI in India\n\nIII. Factors Contributing to the Rise of AI in India\n    A. Government Initiatives and Policies\n    B. Increasing Investments in AI\n    C. Growing Talent Pool\n    D. Rise of Start-ups in the AI sector\n\nIV. Applications of AI in India\n    A. Healthcare\n    B. Agriculture\n    C. Finance and Banking\n    D. Retail Industry\n    E. Education sector\n\nV. Challenges and Opportunities\n    A. Challenges faced by the AI industry in India\n    B. Opportunities for growth and advancement in the AI sector\n    C. Strategies to overcome challenges and leverage opportunities\n\nVI. Future of AI in India\n    A. Predictions for the futur